# HydraMem v2.2 — Graph-Native Temporal Memory

Hack Hydra 2026 · Track 3: Memory + Context Retrieval

This notebook implements a temporal agent memory layer on top of HydraDB.

### Design Goal

Move from:

`HydraDB retrieval → similarity cutoff → newest text wins`

to:

`HydraDB retrieval + Context Graph → typed fact targeting → revision resolution → evidence state → answer / abstain`

HydraDB is the memory substrate **and** the graph. We use:
- **Bring Your Own Graph** at ingestion to declare explicit `SUPERSEDES` edges
- `graph_context=True` at query time to retrieve multi-hop paths and triplets
- Application-level resolver for deterministic `ANSWERABLE / CONFLICTING / NO_EVIDENCE`

> **Security:** Never hard-code an API key. Use Colab Secrets or an environment variable.

### v2.2 hardening changelog
1. `hydra_query` probes the SDK signature once — no per-call exceptions, no warning spam, no unsupported temporal kwargs sent
2. Word-boundary query targeting (no more "deliver"/"believe" false positives)
3. Clean extraction values ("San Francisco", not "San Francisco last week")
4. Same-timestamp contradictions now surface as `CONFLICTING` through the live extraction path
5. Chunk parse failures are logged and counted, never silent
6. `reset_database()` helper for idempotent re-runs
7. Independent hand-authored ground truth in the local benchmark
8. Write-cost benchmark added (judging rubric: read **and** write cost)

## Architecture

```
Conversation / benchmark facts
  ↓
Typed MemoryFact
(subject, predicate, object, time, session, supersedes)
  ↓
HydraDB memory ingest
+ Bring Your Own Graph (explicit SUPERSEDES triplets)
  ↓
HydraDB hybrid query (mode=thinking, graph_context=true)
  ↓
Question → target (subject, predicate)
  ↓
Same-property candidate filtering
  ↓
Revision / supersession resolution
(app logic + HydraDB graph paths)
  ↓
Evidence state
┌──────────────┬──────────────┬──────────────┐
│ ANSWERABLE   │ CONFLICTING  │ NO_EVIDENCE  │
└──────────────┴──────────────┴──────────────┘
```

Timestamp is an attribute used to resolve the state of a specific fact/property.
A newer unrelated fact can never overwrite a different property.

In [ ]:
# Install
!pip -q install "hydradb-sdk>=2,<3" "anthropic>=0.119.0" pydantic pandas scikit-learn

import os
import json
import time
import re
import inspect
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from enum import Enum
from typing import Optional, List, Dict, Any, Tuple

import pandas as pd
from pydantic import BaseModel, Field
from hydra_db import HydraDB

print("HydraDB SDK installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.8/155.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.2 MB/s eta 0:00:00
HydraDB SDK installed.


## 1. Secure HydraDB initialization

In [ ]:
def load_hydra_api_key() -> str:
    try:
        from google.colab import userdata
        key = userdata.get("HYDRA_DB_API_KEY")
        if key:
            return key
    except Exception:
        pass

    key = os.getenv("HYDRA_DB_API_KEY")
    if key:
        return key

    raise RuntimeError(
        "HYDRA_DB_API_KEY is missing. "
        "Add it to Colab Secrets or set it as an environment variable."
    )

API_KEY = load_hydra_api_key()
client = HydraDB(token=API_KEY)

DATABASE = "hydramem_v2_graph_demo"

print("HydraDB client initialized safely.")
print(f"Database: {DATABASE}")

HydraDB client initialized safely.
Database: hydramem_v2_graph_demo


## 2. Typed temporal fact model

A fact is identified by `(subject, predicate)`. Multiple historical values are allowed.
A later fact can explicitly supersede an earlier one via `supersedes_fact_id`.

Examples:
- `(user, lives_in) → New York`
- `(user, lives_in) → San Francisco` *(supersedes previous)*
- `(user, prefers) → dark mode` *(different property → cannot overwrite location)*

In [ ]:
class MemoryState(str, Enum):
    ANSWERABLE = "ANSWERABLE"
    NO_EVIDENCE = "NO_EVIDENCE"
    CONFLICTING = "CONFLICTING"


class MemoryFact(BaseModel):
    fact_id: str
    subject: str
    predicate: str
    object_value: str

    # Event / validity time (not ingestion time)
    timestamp: datetime

    session_id: str
    fact_type: str = "volatile"          # volatile | static | episodic

    # Explicit revision relation (application level)
    supersedes_fact_id: Optional[str] = None

    source_text: Optional[str] = None
    confidence: float = 1.0

    @property
    def property_key(self) -> Tuple[str, str]:
        return (self.subject, self.predicate)

    def to_memory_item(self) -> Dict[str, Any]:
        """Payload for a single memory in context.ingest"""
        return {
            "id": self.fact_id,                       # required for BYOG targeting
            "text": self.source_text or f"{self.subject} {self.predicate} {self.object_value}",
            "infer": False,                           # we supply the graph ourselves
            "metadata": {
                "fact_id": self.fact_id,
                "subject": self.subject,
                "predicate": self.predicate,
                "object_value": self.object_value,
                "timestamp": self.timestamp.isoformat(),
                "session_id": self.session_id,
                "fact_type": self.fact_type,
                "supersedes_fact_id": self.supersedes_fact_id or "",
                "confidence": self.confidence,
                "property_key": f"{self.subject}:{self.predicate}",
            },
        }

    def to_graph_entities_and_relations(self) -> Dict[str, Any]:
        """
        Build a Bring-Your-Own-Graph payload fragment for this fact.
        We create entities for subject & object and a SUPERSEDES edge when present.
        """
        entities = {
            "subj": {
                "name": self.subject,
                "type": "ENTITY",
                "namespace": "memory",
            },
            "obj": {
                "name": self.object_value,
                "type": "VALUE",
                "namespace": "memory",
            },
        }

        relations = [
            {
                "source": "subj",
                "target": "obj",
                "predicate": self.predicate.upper(),
                "context": self.source_text or f"{self.subject} {self.predicate} {self.object_value}",
                "temporal_details": self.timestamp.date().isoformat(),
            }
        ]

        # Explicit supersession edge (the key graph-native signal)
        if self.supersedes_fact_id:
            entities["prev"] = {
                "name": self.supersedes_fact_id,
                "type": "FACT",
                "namespace": "memory",
            }
            relations.append({
                "source": "subj",                      # current fact's subject
                "target": "prev",
                "predicate": "SUPERSEDES",
                "context": f"{self.fact_id} supersedes {self.supersedes_fact_id}",
                "temporal_details": self.timestamp.date().isoformat(),
            })

        return {"entities": entities, "relations": relations}

## 3. Clean demo fixture

Deterministic story used for demos and unit tests:

- Location revision chain: New York → San Francisco → London
- Static preference and job facts
- Separate concurrent-conflict fixture

In [ ]:
BASE = datetime(2026, 8, 1, 10, 0, tzinfo=timezone.utc)

# Kept for reference only: the installed SDK rejects temporal_reasoning /
# temporal_now, and the resolver never reads HydraDB's temporal response fields.
# If a future SDK version re-enables that path, anchor "now" to this fixture's
# timeline instead of the live wall clock.
TEMPORAL_NOW = BASE.isoformat()

def make_fact(
    fact_id: str,
    session_no: int,
    offset_days: int,
    subject: str,
    predicate: str,
    object_value: str,
    *,
    fact_type: str = "volatile",
    supersedes: Optional[str] = None,
    text: Optional[str] = None,
    confidence: float = 1.0,
) -> MemoryFact:
    return MemoryFact(
        fact_id=fact_id,
        subject=subject,
        predicate=predicate,
        object_value=object_value,
        timestamp=BASE + timedelta(days=offset_days),
        session_id=f"sess_{session_no:03d}",
        fact_type=fact_type,
        supersedes_fact_id=supersedes,
        source_text=text,
        confidence=confidence,
    )

# ------------------------------------------------------------
# CLEAN DEMO: normal temporal history
# ------------------------------------------------------------
facts = [
    make_fact(
        "f_location_nyc", 1, 0,
        "user", "lives_in", "New York",
        text="I live in New York."
    ),
    make_fact(
        "f_pref_dark", 1, 1,
        "user", "prefers", "dark mode",
        fact_type="static",
        text="I prefer dark mode for my apps."
    ),
    make_fact(
        "f_location_sf", 2, 4,
        "user", "lives_in", "San Francisco",
        supersedes="f_location_nyc",
        text="I moved to San Francisco last week."
    ),
    make_fact(
        "f_job_engineer", 3, 7,
        "user", "job", "software engineer",
        fact_type="static",
        text="I work as a software engineer."
    ),
    make_fact(
        "f_location_london", 4, 10,
        "user", "lives_in", "London",
        supersedes="f_location_sf",
        text="I relocated to London."
    ),
]

# ------------------------------------------------------------
# CONFLICT FIXTURE: two contradictory values at same timestamp
# ------------------------------------------------------------
conflict_facts = [
    make_fact(
        "f_location_boston_a", 5, 12,
        "user", "lives_in", "Boston",
        text="I live in Boston now."
    ),
    make_fact(
        "f_location_boston_b", 6, 12,
        "user", "lives_in", "Cambridge",
        text="I actually live in Cambridge now."
    ),
]

fact_df = pd.DataFrame([f.model_dump() for f in facts])
fact_df[["fact_id", "subject", "predicate", "object_value", "timestamp", "session_id", "supersedes_fact_id"]]

,fact_id,subject,predicate,object_value,timestamp,session_id,supersedes_fact_id
0,f_location_nyc,user,lives_in,New York,2026-08-01 10:00:00+00:00,sess_001,None
1,f_pref_dark,user,prefers,dark mode,2026-08-02 10:00:00+00:00,sess_001,None
2,f_location_sf,user,lives_in,San Francisco,2026-08-05 10:00:00+00:00,sess_002,f_location_nyc
3,f_job_engineer,user,job,software engineer,2026-08-08 10:00:00+00:00,sess_003,None
4,f_location_london,user,lives_in,London,2026-08-11 10:00:00+00:00,sess_004,f_location_sf


## 4. Revision-chain resolution (application layer)

Resolves **only** facts that match the requested `(subject, predicate)`.

Rules:
1. No candidates → `NO_EVIDENCE`
2. Multiple terminal facts with different values at the same timestamp → `CONFLICTING`
3. Explicit supersession creates a revision chain; the terminal fact wins
4. Unrelated newer facts (different predicate) are ignored

In [ ]:
def resolve_revision_state(
    candidate_facts: List[MemoryFact],
    subject: str,
    predicate: str,
) -> Dict[str, Any]:
    relevant = [
        f for f in candidate_facts
        if f.subject == subject and f.predicate == predicate
    ]

    if not relevant:
        return {
            "state": MemoryState.NO_EVIDENCE.value,
            "fact": None,
            "chain": [],
            "reason": "No fact exists for the requested subject/property.",
        }

    by_id = {f.fact_id: f for f in relevant}
    superseded_ids = {
        f.supersedes_fact_id for f in relevant if f.supersedes_fact_id
    }

    current = [f for f in relevant if f.fact_id not in superseded_ids]

    if not current:
        return {
            "state": MemoryState.CONFLICTING.value,
            "fact": None,
            "chain": sorted(relevant, key=lambda f: f.timestamp),
            "reason": "Revision graph has no terminal fact.",
        }

    newest_ts = max(f.timestamp for f in current)
    newest = [f for f in current if f.timestamp == newest_ts]

    if len(newest) > 1:
        distinct_values = {f.object_value for f in newest}
        if len(distinct_values) > 1:
            return {
                "state": MemoryState.CONFLICTING.value,
                "fact": None,
                "chain": sorted(relevant, key=lambda f: f.timestamp),
                "reason": "Multiple current contradictory values exist at the same timestamp.",
            }

    winner = sorted(current, key=lambda f: (f.timestamp, f.confidence), reverse=True)[0]

    # Reconstruct revision chain backwards
    chain = []
    cursor = winner
    seen = set()
    while cursor and cursor.fact_id not in seen:
        seen.add(cursor.fact_id)
        chain.append(cursor)
        if not cursor.supersedes_fact_id:
            break
        cursor = by_id.get(cursor.supersedes_fact_id)

    return {
        "state": MemoryState.ANSWERABLE.value,
        "fact": winner,
        "chain": list(reversed(chain)),
        "reason": "Resolved from the terminal fact in the revision chain.",
    }


def pretty_resolution(result: Dict[str, Any]) -> None:
    print(f"State : {result['state']}")
    print(f"Reason: {result['reason']}")
    if result["fact"]:
        f = result["fact"]
        print(f"Answer: {f.object_value}")
        print(f"Fact ID: {f.fact_id}")
        print(f"Timestamp: {f.timestamp.isoformat()}")
        print("Revision chain:")
        for item in result["chain"]:
            print(f"  {item.fact_id}: {item.object_value} @ {item.timestamp.date()}")

In [ ]:
print("=== TEST A: Temporal revision resolution ===")
location_resolution = resolve_revision_state(facts, "user", "lives_in")
pretty_resolution(location_resolution)
print("Expected: ANSWERABLE → London")
print("PASS ✅" if (
    location_resolution["state"] == MemoryState.ANSWERABLE.value
    and location_resolution["fact"].object_value == "London"
) else "FAIL ❌")

print("\n=== TEST B: Missing fact / abstention ===")
missing_resolution = resolve_revision_state(facts, "user", "favorite_pet")
pretty_resolution(missing_resolution)
print("Expected: NO_EVIDENCE")
print("PASS ✅" if missing_resolution["state"] == MemoryState.NO_EVIDENCE.value else "FAIL ❌")

print("\n=== TEST C: Concurrent conflict ===")
conflict_resolution = resolve_revision_state(conflict_facts, "user", "lives_in")
pretty_resolution(conflict_resolution)
print("Expected: CONFLICTING")
print("PASS ✅" if conflict_resolution["state"] == MemoryState.CONFLICTING.value else "FAIL ❌")

=== TEST A: Temporal revision resolution ===
State : ANSWERABLE
Reason: Resolved from the terminal fact in the revision chain.
Answer: London
Fact ID: f_location_london
Timestamp: 2026-08-11T10:00:00+00:00
Revision chain:
  f_location_nyc: New York @ 2026-08-01
  f_location_sf: San Francisco @ 2026-08-05
  f_location_london: London @ 2026-08-11
Expected: ANSWERABLE → London
PASS ✅

=== TEST B: Missing fact / abstention ===
State : NO_EVIDENCE
Reason: No fact exists for the requested subject/property.
Expected: NO_EVIDENCE
PASS ✅

=== TEST C: Concurrent conflict ===
State : CONFLICTING
Reason: Multiple current contradictory values exist at the same timestamp.
Expected: CONFLICTING
PASS ✅


## 5. HydraDB ingestion with Bring Your Own Graph

We declare explicit `SUPERSEDES` triplets at ingest time so the Context Graph contains deterministic temporal edges.

See: [Bring Your Own Graph](https://docs.hydradb.com/essentials/v2/bring-your-own-graph) and [Context Graphs](https://docs.hydradb.com/essentials/v2/context-graphs).

In [ ]:
def ensure_database_ready(database: str, poll_seconds: int = 3, timeout_seconds: int = 180) -> None:
    try:
        client.databases.create(database=database)
    except Exception as exc:
        print(f"Database create note: {exc}")

    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        infra = client.databases.status(database=database).data.infra
        if infra.ready_for_ingestion:
            print(f"Database ready: {database}")
            return
        time.sleep(poll_seconds)

    raise TimeoutError(f"Database '{database}' was not ready for ingestion within {timeout_seconds}s.")


def reset_database(database: str) -> None:
    """Best-effort wipe + recreate so repeated runs don't accumulate duplicate memories."""
    try:
        client.databases.delete(database=database)
        print(f"Deleted existing database: {database}")
    except Exception as exc:
        print(f"Delete skipped ({exc}). If this SDK lacks databases.delete, use a fresh DATABASE name instead.")
    ensure_database_ready(database)


def ingest_facts_with_graph(facts_to_ingest: List[MemoryFact], database: str) -> List[str]:
    """
    Ingest memories and attach a Bring-Your-Own-Graph payload
    that declares SUPERSEDES edges for temporal revision.
    """
    memories = [f.to_memory_item() for f in facts_to_ingest]

    # Build graph_payload keyed by memory id
    graph_payload = {}
    for f in facts_to_ingest:
        graph_payload[f.fact_id] = f.to_graph_entities_and_relations()

    response = client.context.ingest(
        type="memory",
        database=database,
        memories=json.dumps(memories),
        graph_payload=json.dumps(graph_payload),
    )

    ids = [r.id for r in response.data.results]
    print(f"Ingested {len(ids)} memories with BYOG graph.")
    return ids


def wait_for_indexing(ids: List[str], database: str, poll_seconds: int = 2, timeout_seconds: int = 180) -> None:
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        statuses = client.context.status(database=database, ids=ids).data.statuses
        if any(s.indexing_status == "errored" for s in statuses):
            errors = [getattr(s, "error_message", "unknown") for s in statuses if s.indexing_status == "errored"]
            raise RuntimeError(f"HydraDB indexing error(s): {errors}")

        if all(s.indexing_status == "completed" for s in statuses):
            print("Indexing completed (graph ready).")
            return

        time.sleep(poll_seconds)

    raise TimeoutError("HydraDB indexing did not complete before timeout.")

In [ ]:
# Create / prepare the demo database and ingest the clean temporal history.
# If you are re-running the notebook and want a clean slate, run this first:
#     reset_database(DATABASE)
ensure_database_ready(DATABASE)

ingested_ids = ingest_facts_with_graph(facts, DATABASE)
wait_for_indexing(ingested_ids, DATABASE)

print(f"\nReady. Database: {DATABASE}")

Database create note: headers: {'content-type': 'application/json; charset=utf-8', 'traceparent': '00-ad4fb2b8a2fa666d84288e3719360825-d8054f768b3baf47-01', 'x-request-id': '501f269f-0888-4ac7-89ff-ff317e4b0c07', 'date': 'Mon, 17 Aug 2026 18:16:53 GMT', 'content-length': '317', 'x-app-name': 'hdb-go'}, status_code: 409, body: data=None detail=HandlerErrorDetail(deprecated=None, deprecated_field=None, error_code='DATABASE_ALREADY_EXISTS', message='Database ID already exists', preferred_field=None, success=False) error=HandlerApiError(code='DATABASE_ALREADY_EXISTS', message='Database ID already exists') meta=HandlerErrorMeta(api_version='2.0.1', latency_ms=44.2, request_id='501f269f-0888-4ac7-89ff-ff317e4b0c07') success=False
Database ready: hydramem_v2_graph_demo
Ingested 5 memories with BYOG graph.
Indexing completed (graph ready).

Ready. Database: hydramem_v2_graph_demo


## 6. Query path — HydraDB retrieval + Context Graph + temporal resolution

Flow:
1. Hybrid query with `mode="thinking"` and `graph_context=True`
2. Convert retrieved chunks back into `MemoryFact`s using metadata
3. Resolve revision state with application logic
4. Surface HydraDB graph triplets (`query_paths` / `chunk_relations`) alongside the answer

**v2.2 hardening:**
- `hydra_query` probes the SDK signature once and filters kwargs up front — no exception/retry in the hot path, no warning spam
- `temporal_reasoning` / `temporal_now` are no longer sent: the installed SDK rejects them and the resolver never reads those response fields
- Chunk parse failures are logged and counted, never silent
- All query-target cues are word-boundary anchored

In [ ]:
def obj_get(obj: Any, key: str, default: Any = None) -> Any:
    if obj is None:
        return default
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


# ---------------------------------------------------------------------------
# Query wrapper: probes the installed SDK signature ONCE, then filters kwargs.
# No exceptions in the hot path, no per-call warnings.
# ---------------------------------------------------------------------------
_QUERY_SIG_PARAMS = None

def hydra_query(client_obj, **kwargs):
    """Call client.query with only the kwargs the installed SDK actually accepts."""
    global _QUERY_SIG_PARAMS
    if _QUERY_SIG_PARAMS is None:
        try:
            _QUERY_SIG_PARAMS = set(inspect.signature(client_obj.query).parameters.keys())
        except (TypeError, ValueError):
            _QUERY_SIG_PARAMS = set()  # can't introspect -- send everything
    if _QUERY_SIG_PARAMS:
        dropped = set(kwargs) - _QUERY_SIG_PARAMS
        if dropped:
            print(f"  [hydra_query] installed SDK lacks {sorted(dropped)}; omitting (printed once).")
        kwargs = {k: v for k, v in kwargs.items() if k in _QUERY_SIG_PARAMS}
    return client_obj.query(**kwargs)


# ---------------------------------------------------------------------------
# Visible chunk-drop accounting (no silent failures)
# ---------------------------------------------------------------------------
_CHUNK_DROP_COUNT = 0

def _record_chunk_drop(reason: str, chunk_id: Any) -> None:
    global _CHUNK_DROP_COUNT
    _CHUNK_DROP_COUNT += 1
    print(f"  [hydra_chunk_to_fact] dropped chunk {chunk_id!r}: {reason} "
          f"(total dropped: {_CHUNK_DROP_COUNT})")

def reset_chunk_drop_count() -> None:
    global _CHUNK_DROP_COUNT
    _CHUNK_DROP_COUNT = 0

def get_chunk_drop_count() -> int:
    return _CHUNK_DROP_COUNT


def hydra_chunk_to_fact(chunk: Any) -> Optional[MemoryFact]:
    """Convert a retrieved HydraDB chunk into a MemoryFact. Drops are logged, never silent."""
    metadata = obj_get(chunk, "metadata", {}) or {}
    if not isinstance(metadata, dict):
        metadata = {}

    # Some responses nest metadata under additional_metadata
    additional = obj_get(chunk, "additional_metadata", {}) or {}
    if isinstance(additional, dict):
        metadata = {**metadata, **additional}

    fact_id = metadata.get("fact_id") or obj_get(chunk, "id") or obj_get(chunk, "chunk_uuid")
    fields = {
        "fact_id": fact_id,
        "subject": metadata.get("subject"),
        "predicate": metadata.get("predicate"),
        "object_value": metadata.get("object_value"),
        "timestamp": metadata.get("timestamp"),
        "session_id": metadata.get("session_id"),
    }
    missing = [k for k, v in fields.items() if not v]
    if missing:
        _record_chunk_drop(f"missing metadata fields {missing}", fact_id)
        return None

    try:
        timestamp = fields["timestamp"]
        if isinstance(timestamp, str):
            timestamp = datetime.fromisoformat(timestamp.replace("Z", "+00:00"))
        supersedes = metadata.get("supersedes_fact_id") or None
        if supersedes == "":
            supersedes = None
        return MemoryFact(
            fact_id=str(fields["fact_id"]),
            subject=fields["subject"],
            predicate=fields["predicate"],
            object_value=fields["object_value"],
            timestamp=timestamp,
            session_id=fields["session_id"],
            fact_type=metadata.get("fact_type", "volatile"),
            supersedes_fact_id=supersedes,
            source_text=obj_get(chunk, "chunk_content") or obj_get(chunk, "content"),
            confidence=float(metadata.get("confidence", 1.0)),
        )
    except Exception as e:
        _record_chunk_drop(f"parse error: {e}", fact_id)
        return None


# ---------------------------------------------------------------------------
# Word-boundary target extraction (fixes "deliver"/"believe" false positives)
# ---------------------------------------------------------------------------
_LOCATION_RE = re.compile(
    r"\b(?:live|lives|living|location|where\s+does|where\s+is|city|moved|relocated|"
    r"reside|resides|based\s+in|currently\s+in|hometown|home\s+town)\b"
)
_JOB_RE = re.compile(
    r"\b(?:job|work\s+as|works\s+as|working\s+as|profession|occupation|employed|"
    r"employment|career|job\s+title|role\s+at|role|title)\b"
)
_PREF_RE = re.compile(
    r"\b(?:prefer|prefers|preference|preferences|like|likes|dark\s+mode|light\s+mode|theme)\b"
)
_RELATION_RE = re.compile(r"\b(?:manager|boss|reports?\s+to)\b")
_TEAM_RE = re.compile(r"\b(?:teammate|colleague|collaborat\w*)\b")
_SALARY_RE = re.compile(r"\b(?:salary|pay|compensation|income|make|earn|earning)\b")
_OWN_RE = re.compile(r"\b(?:own|owns|owned|ownership)\b")


def extract_target_from_query(query: str) -> Tuple[str, str]:
    """
    Rule-based target extractor with open-predicate support.
    All cues are word-boundary anchored.
    """
    q = query.lower().strip()

    # ---- most-specific open patterns first ----
    m = re.search(
        r"\b(?:favorite|favourite)\s+([a-z][a-z0-9\-]*(?:\s+[a-z][a-z0-9\-]*)?)\b",
        q,
    )
    if m:
        slot = re.sub(r"[\s\-]+", " ", m.group(1).strip())
        if slot in {"pet", "dog", "cat", "puppy", "kitten"}:
            return "user", "has_pet"
        if slot in {"dark_mode", "light_mode", "theme", "mode"}:
            return "user", "prefers"
        return "user", f"favorite_{slot}"

    # ---- pet / animal ownership ----
    if re.search(r"\b(?:pet|dog|cat|puppy|kitten)\b", q):
        return "user", "has_pet"

    # ---- location ----
    if _LOCATION_RE.search(q):
        return "user", "lives_in"

    # ---- job / occupation ----
    if _JOB_RE.search(q) and not _RELATION_RE.search(q) and not _TEAM_RE.search(q):
        return "user", "job"

    # ---- salary / compensation ----
    if _SALARY_RE.search(q):
        return "user", "salary"

    # ---- ownership ----
    if _OWN_RE.search(q):
        return "user", "owns"

    # ---- core preference / theme ----
    if _PREF_RE.search(q):
        return "user", "prefers"

    # ---- direct relationship queries ----
    if _RELATION_RE.search(q) and not _LOCATION_RE.search(q) and not _JOB_RE.search(q):
        return "user", "reports_to"

    if _TEAM_RE.search(q) and not _LOCATION_RE.search(q) and not _JOB_RE.search(q):
        return "user", "collaborates_with"

    return "user", "unknown_property"


def answer_memory_query(
    query: str,
    database: str,
    top_k: int = 20,
    min_relevancy: float = 0.0,
) -> Dict[str, Any]:
    """
    Full graph-native answer path.
    """
    subject, predicate = extract_target_from_query(query)

    reset_chunk_drop_count()
    result = hydra_query(
        client,
        database=database,
        query=query,
        type="memory",
        query_by="hybrid",
        mode="thinking",
        max_results=top_k,
        graph_context=True,          # ← key flag from Context Graphs docs
        # temporal_reasoning / temporal_now intentionally removed: the installed
        # SDK rejects them, and the resolver never reads HydraDB's temporal
        # response fields -- revision truth comes from the deterministic
        # SUPERSEDES chain (Section 4). One clean API call per query.
    )

    chunks = obj_get(result.data, "chunks", []) or []
    graph_ctx = obj_get(result.data, "graph_context", {}) or {}

    # Convert to typed facts
    candidate_facts = []
    for ch in chunks:
        fact = hydra_chunk_to_fact(ch)
        if fact:
            candidate_facts.append(fact)
    if get_chunk_drop_count():
        print(f"  [answer] WARNING: {get_chunk_drop_count()} retrieved chunk(s) "
              f"could not be parsed as facts -- see drop log above.")

    # Application-level temporal resolution
    resolution = resolve_revision_state(candidate_facts, subject, predicate)

    # Package answer + graph evidence
    answer = None
    if resolution["state"] == MemoryState.ANSWERABLE.value and resolution["fact"]:
        answer = resolution["fact"].object_value

    return {
        "query": query,
        "target": f"{subject}::{predicate}",
        "state": resolution["state"],
        "answer": answer,
        "reason": resolution["reason"],
        "revision_chain": [
            {"fact_id": f.fact_id, "value": f.object_value, "ts": f.timestamp.isoformat()}
            for f in resolution["chain"]
        ],
        "retrieved_candidates": len(candidate_facts),
        "graph_context": {
            "query_paths": obj_get(graph_ctx, "query_paths", []),
            "chunk_relations": obj_get(graph_ctx, "chunk_relations", []),
            "chunk_id_to_group_ids": obj_get(graph_ctx, "chunk_id_to_group_ids", {}),
        },
    }


def show_memory_answer(result: Dict[str, Any]) -> None:
    print(f"Query   : {result['query']}")
    print(f"Target  : {result['target']}")
    print(f"State   : {result['state']}")
    print(f"Answer  : {result['answer']}")
    print(f"Reason  : {result['reason']}")
    print(f"Candidates retrieved: {result['retrieved_candidates']}")

    if result["revision_chain"]:
        print("\nRevision chain (application):")
        for step in result["revision_chain"]:
            print(f"  {step['fact_id']}: {step['value']} @ {step['ts'][:10]}")

    # ---- HydraDB Context Graph evidence ----
    graph = result.get("graph_context", {})
    paths = graph.get("query_paths") or []
    rels = graph.get("chunk_relations") or []

    print("\nHydraDB graph evidence:")
    print(f"  query_paths     : {len(paths)}")
    print(f"  chunk_relations : {len(rels)}")

    def print_triplets(triplets, indent="   "):
        for t in triplets:
            src = obj_get(obj_get(t, "source"), "name", "?")
            rel = obj_get(obj_get(t, "relation"), "canonical_predicate", "?")
            tgt = obj_get(obj_get(t, "target"), "name", "?")
            print(f"{indent}[{src}] --{rel}--> [{tgt}]")

    if paths:
        print("\nquery_paths triplets:")
        for p in paths[:5]:
            print_triplets(obj_get(p, "triplets", []) or [])

    if rels:
        print("\nchunk_relations triplets:")
        for r in rels[:8]:   # show up to 8
            triplets = obj_get(r, "triplets", None)
            if triplets:
                print_triplets(triplets)
            else:
                src = obj_get(obj_get(r, "source"), "name", "?")
                rel = obj_get(obj_get(r, "relation"), "canonical_predicate",
                              obj_get(r, "predicate", "?"))
                tgt = obj_get(obj_get(r, "target"), "name", "?")
                print(f"  [{src}] --{rel}--> [{tgt}]")

## 7. Live demo queries

These three cases are the core of the 3-minute video:

1. **Revision** – NYC → SF → London → current location is London
2. **Property isolation** – a newer unrelated fact must not overwrite `lives_in`
3. **Abstention** – missing fact returns `NO_EVIDENCE`

In [ ]:
print("=" * 60)
print("DEMO 1 — Temporal revision")
print("=" * 60)
result1 = answer_memory_query("Where does the user live?", DATABASE)
show_memory_answer(result1)

DEMO 1 — Temporal revision
Query   : Where does the user live?
Target  : user::lives_in
State   : ANSWERABLE
Answer  : London
Reason  : Resolved from the terminal fact in the revision chain.
Candidates retrieved: 13

Revision chain (application):
  f_location_nyc: New York @ 2026-08-01
  f_location_sf: San Francisco @ 2026-08-05
  f_location_london: London @ 2026-08-11

HydraDB graph evidence:
  query_paths     : 0
  chunk_relations : 15

chunk_relations triplets:
   [user] --LIVES_IN--> [new york]
   [manager_alex] --LIVES_IN--> [berlin]
   [user] --LIVES_IN--> [london]
   [user] --REPORTS_TO--> [manager_alex]
   [user] --SUPERSEDES--> [f_location_sf]
   [user] --SUPERSEDES--> [f_location_nyc]
   [user] --JOB--> [software engineer]
   [user] --BENCHMARKED_IN--> [city_0]


In [ ]:
print("=" * 60)
print("DEMO 2 — Static preference (property isolation)")
print("=" * 60)
result2 = answer_memory_query("What does the user prefer?", DATABASE)
show_memory_answer(result2)

DEMO 2 — Static preference (property isolation)
Query   : What does the user prefer?
Target  : user::prefers
State   : ANSWERABLE
Answer  : dark mode
Reason  : Resolved from the terminal fact in the revision chain.
Candidates retrieved: 13

Revision chain (application):
  f_pref_dark: dark mode @ 2026-08-02

HydraDB graph evidence:
  query_paths     : 0
  chunk_relations : 15

chunk_relations triplets:
   [user] --PREFERS--> [dark mode]
   [user] --LIVES_IN--> [london]
   [user] --REPORTS_TO--> [manager_alex]
   [user] --SUPERSEDES--> [f_location_sf]
   [user] --SUPERSEDES--> [f_location_nyc]
   [user] --JOB--> [software engineer]
   [user] --LIVES_IN--> [new york]
   [user] --BENCHMARKED_IN--> [city_0]


In [ ]:
print("=" * 60)
print("DEMO 3 — Correct abstention")
print("=" * 60)
result3 = answer_memory_query("What is the user's favorite pet?", DATABASE)
show_memory_answer(result3)

DEMO 3 — Correct abstention
Query   : What is the user's favorite pet?
Target  : user::has_pet
State   : NO_EVIDENCE
Answer  : None
Reason  : No fact exists for the requested subject/property.
Candidates retrieved: 13

HydraDB graph evidence:
  query_paths     : 0
  chunk_relations : 15

chunk_relations triplets:
   [user] --LIVES_IN--> [london]
   [user] --REPORTS_TO--> [manager_alex]
   [user] --SUPERSEDES--> [f_location_sf]
   [user] --SUPERSEDES--> [f_location_nyc]
   [user] --JOB--> [software engineer]
   [user] --LIVES_IN--> [new york]
   [user] --BENCHMARKED_IN--> [city_0]
   [user] --BENCHMARKED_IN--> [city_1]


## 8. Synthetic multi-session benchmark (local unit tests)

Generates ~35 sessions with location revisions, job changes, static preferences and noise.
Used as a fast regression suite before running real LongMemEval-V2 / BEAM samples.

**v2.2:** expected values for the temporal cases are hand-authored ground truth (`Boston`, `researcher`), deliberately *not* computed with `max(timestamp)` — the oracle no longer shares the resolver's latest-wins rule.

In [ ]:
def generate_synthetic_sessions(n_sessions: int = 35, seed: int = 7) -> List[MemoryFact]:
    import random
    rng = random.Random(seed)

    cities = ["New York", "San Francisco", "London", "Boston", "Chicago", "Seattle"]
    jobs   = ["software engineer", "designer", "product manager", "researcher"]

    generated = []
    location_fact_id = None
    job_fact_id = None

    for i in range(1, n_sessions + 1):
        session_date = BASE + timedelta(days=i)

        # Episodic noise
        generated.append(MemoryFact(
            fact_id=f"noise_{i:03d}",
            subject="user",
            predicate="commented_on",
            object_value=f"the weather on day {i}",
            timestamp=session_date,
            session_id=f"sess_{i:03d}",
            fact_type="episodic",
            source_text=f"We talked about the weather on day {i}.",
        ))

        # Location revisions
        if i in {1, 9, 18, 27}:
            city = cities[(i // 9) % len(cities)]
            new_id = f"location_{i:03d}"
            generated.append(MemoryFact(
                fact_id=new_id,
                subject="user",
                predicate="lives_in",
                object_value=city,
                timestamp=session_date,
                session_id=f"sess_{i:03d}",
                fact_type="volatile",
                supersedes_fact_id=location_fact_id,
                source_text=f"I now live in {city}.",
            ))
            location_fact_id = new_id

        # Job revision
        if i in {3, 22}:
            job = "software engineer" if i == 3 else "researcher"
            new_id = f"job_{i:03d}"
            generated.append(MemoryFact(
                fact_id=new_id,
                subject="user",
                predicate="job",
                object_value=job,
                timestamp=session_date,
                session_id=f"sess_{i:03d}",
                fact_type="volatile",
                supersedes_fact_id=job_fact_id,
                source_text=f"My job is {job}.",
            ))
            job_fact_id = new_id

        # Static preference
        if i == 5:
            generated.append(MemoryFact(
                fact_id="preference_005",
                subject="user",
                predicate="prefers",
                object_value="dark mode",
                timestamp=session_date,
                session_id=f"sess_{i:03d}",
                fact_type="static",
                source_text="I prefer dark mode.",
            ))

    return generated


benchmark_facts = generate_synthetic_sessions(n_sessions=35)
print("Sessions:", len({f.session_id for f in benchmark_facts}))
print("Facts   :", len(benchmark_facts))

Sessions: 35
Facts   : 42


In [ ]:
def local_answer_memory_query(query: str, corpus: List[MemoryFact]) -> Dict[str, Any]:
    """Pure local version of the answer path (no HydraDB call)."""
    subject, predicate = extract_target_from_query(query)
    resolution = resolve_revision_state(corpus, subject, predicate)
    answer = None
    if resolution["state"] == MemoryState.ANSWERABLE.value and resolution["fact"]:
        answer = resolution["fact"].object_value
    return {
        "answer": answer,
        "state": resolution["state"],
        "reason": resolution["reason"],
    }


def run_local_benchmark(facts: List[MemoryFact]) -> pd.DataFrame:
    benchmark_conflict = [
        make_fact("bench_conflict_a", 100, 50, "user", "lives_in", "Boston"),
        make_fact("bench_conflict_b", 101, 50, "user", "lives_in", "Cambridge"),
    ]

    # Explicit open-predicate fixtures prove that the loosened schema is actually queryable.
    open_facts = facts + [
        make_fact("favorite_color_blue", 200, 20, "user", "favorite_color", "blue", fact_type="static"),
        make_fact("pet_max", 201, 21, "user", "has_pet", "Max", fact_type="static"),
        make_fact("salary_180k", 202, 22, "user", "salary", "$180k", fact_type="static"),
    ]

    # NOTE: "Boston" and "researcher" are hand-authored ground truth for the
    # deterministic seed=7 generator (location revisions at sessions 1/9/18/27 ->
    # New York, San Francisco, London, Boston; job revisions at 3/22 -> engineer,
    # researcher). They are deliberately NOT computed with max(timestamp) so the
    # oracle does not share the resolver's latest-wins rule.
    cases = [
        {"name": "current location", "query": "Where does the user live?",
         "expected": "Boston",
         "type": "temporal", "corpus": facts},
        {"name": "current job", "query": "What is the user's job?",
         "expected": "researcher",
         "type": "overwrite", "corpus": facts},
        {"name": "static preference", "query": "What does the user prefer?",
         "expected": "dark mode", "type": "static", "corpus": facts},
        {"name": "missing fact abstention", "query": "What is the user's favorite pet?",
         "expected": None, "type": "abstention", "corpus": facts},
        {"name": "property isolation", "query": "Where does the user live?",
         "expected": "Boston",
         "type": "property-isolation",
         "corpus": facts + [make_fact("newer_noise", 90, 100, "user", "commented_on", "a new weather event")]},
        {"name": "explicit concurrent conflict", "query": "Where does the user live?",
         "expected": None, "type": "conflict", "corpus": benchmark_conflict},
        {"name": "revision chain intact", "query": "Where does the user live?",
         "expected": "Boston",
         "type": "revision-chain", "corpus": facts},
        {"name": "unknown property abstention", "query": "What is the user's favorite pet?",
         "expected": None, "type": "no-evidence", "corpus": facts},
        {"name": "open favorite color", "query": "What is the user's favorite color?",
         "expected": "blue", "type": "open-predicate", "corpus": open_facts},
        {"name": "open pet property", "query": "What pet does the user have?",
         "expected": "Max", "type": "open-predicate", "corpus": open_facts},
        {"name": "open salary property", "query": "What is the user's salary?",
         "expected": "$180k", "type": "open-predicate", "corpus": open_facts},
    ]

    rows = []
    for case in cases:
        subject, predicate = extract_target_from_query(case["query"])
        result = local_answer_memory_query(case["query"], case["corpus"])
        if case["type"] == "conflict":
            passed = result["state"] == MemoryState.CONFLICTING.value
        elif case["expected"] is None:
            passed = result["state"] == MemoryState.NO_EVIDENCE.value
        else:
            passed = (result["answer"] == case["expected"]
                      and result["state"] == MemoryState.ANSWERABLE.value)

        rows.append({
            "case": case["name"],
            "type": case["type"],
            "target": f"{subject}::{predicate}",
            "expected": case["expected"],
            "actual": result["answer"],
            "state": result["state"],
            "passed": passed,
        })

    return pd.DataFrame(rows)


benchmark_results = run_local_benchmark(benchmark_facts)
display(benchmark_results)

summary = {
    "overall_accuracy": float(benchmark_results["passed"].mean()),
    "total_cases": int(len(benchmark_results)),
    "passed_cases": int(benchmark_results["passed"].sum()),
    "failed_cases": int((~benchmark_results["passed"]).sum()),
    "sessions": len({f.session_id for f in benchmark_facts}),
    "facts": len(benchmark_facts),
}
print(json.dumps(summary, indent=2))
print("\nResults by test type:")
display(benchmark_results.groupby("type")["passed"].mean().round(3))

assert benchmark_results["passed"].all(), "Local benchmark has failing regression cases."
print("\nAll local regression cases PASS ✅")

,case,type,target,expected,actual,state,passed
0,current location,temporal,user::lives_in,Boston,Boston,ANSWERABLE,True
1,current job,overwrite,user::job,researcher,researcher,ANSWERABLE,True
2,static preference,static,user::prefers,dark mode,dark mode,ANSWERABLE,True
3,missing fact abstention,abstention,user::has_pet,None,None,NO_EVIDENCE,True
4,property isolation,property-isolation,user::lives_in,Boston,Boston,ANSWERABLE,True
5,explicit concurrent conflict,conflict,user::lives_in,None,None,CONFLICTING,True
6,revision chain intact,revision-chain,user::lives_in,Boston,Boston,ANSWERABLE,True
7,unknown property abstention,no-evidence,user::has_pet,None,None,NO_EVIDENCE,True
8,open favorite color,open-predicate,user::favorite_color,blue,blue,ANSWERABLE,True
9,open pet property,open-predicate,user::has_pet,Max,Max,ANSWERABLE,True


{
  "overall_accuracy": 1.0,
  "total_cases": 11,
  "passed_cases": 11,
  "failed_cases": 0,
  "sessions": 35,
  "facts": 42
}

Results by test type:


,passed
type,
abstention,1.0
conflict,1.0
no-evidence,1.0
open-predicate,1.0
overwrite,1.0
property-isolation,1.0
revision-chain,1.0
static,1.0
temporal,1.0



All local regression cases PASS ✅


## 9. Multi-hop relationship traversal

Everything so far resolves *one entity's own property over time*. This section adds a second axis: resolving a question that requires crossing an **entity boundary** — e.g. "Where does the user's manager live?" requires two hops:

1. `(user, reports_to) → manager_alex` — a relation edge (itself revisable)
2. `(manager_alex, lives_in) → Tokyo` — a normal revision chain, but on a *different subject*

No single retrieved chunk contains the full answer — "Alex relocated to Tokyo" never mentions the user, and "I report to Alex" never mentions a city. A pure similarity match against the question text has no structural way to compose these two facts; answering correctly requires walking an edge in the graph. This directly targets "an interesting use of relationships, traversal or context" from the judging rubric.

Reuses `to_graph_entities_and_relations()` unmodified — relation edges are just facts whose `object_value` is another entity's name, so they ingest through the existing BYOG pipeline with no schema changes.

In [ ]:
def resolve_multi_hop(
    candidate_facts: List[MemoryFact],
    subject: str,
    relation_predicate: str,
    target_predicate: str,
) -> Dict[str, Any]:
    """
    Two-hop resolution:
      1. Resolve the relation edge (subject, relation_predicate) -> related_entity.
         Relation edges are themselves revisable (e.g. reassigned to a new manager),
         so they go through the SAME resolver as any other fact.
      2. Resolve (related_entity, target_predicate) with the normal revision-chain
         resolver, now scoped to the related entity instead of the original subject.
    Returns a combined evidence trail spanning both hops.
    """
    hop1 = resolve_revision_state(candidate_facts, subject, relation_predicate)
    if hop1["state"] != MemoryState.ANSWERABLE.value:
        return {
            "state": hop1["state"],
            "fact": None,
            "hops": [hop1],
            "related_entity": None,
            "reason": f"Could not resolve hop 1 ({subject}.{relation_predicate}): {hop1['reason']}",
        }

    related_entity = hop1["fact"].object_value
    hop2 = resolve_revision_state(candidate_facts, related_entity, target_predicate)

    return {
        "state": hop2["state"],
        "fact": hop2["fact"],
        "hops": [hop1, hop2],
        "related_entity": related_entity,
        "reason": (
            f"Hop 1 resolved '{subject}.{relation_predicate}' -> '{related_entity}'; "
            f"Hop 2 resolved '{related_entity}.{target_predicate}': {hop2['reason']}"
        ),
    }


def extract_multi_hop_target(query: str) -> Optional[Tuple[str, str, str]]:
    """
    Lightweight two-hop target matcher for the demo.
    Relation side stays on the explicit core relation vocabulary; target side
    also supports open properties such as favorite_* / has_pet / salary / owns.
    """
    q = query.lower().strip()
    relation_map = {
        "manager": "reports_to",
        "boss": "reports_to",
        "teammate": "collaborates_with",
        "colleague": "collaborates_with",
    }
    relation_hit = next((v for k, v in relation_map.items() if re.search(rf"\b{k}\b", q)), None)
    if not relation_hit:
        return None

    m = re.search(
        r"\b(?:favorite|favourite)\s+([a-z][a-z0-9\-]*(?:\s+[a-z][a-z0-9\-]*)?)\b",
        q,
    )
    if m:
        slot = re.sub(r"[\s\-]+", " ", m.group(1).strip())
        if slot in {"pet", "dog", "cat", "puppy", "kitten"}:
            return "user", relation_hit, "has_pet"
        if slot in {"dark_mode", "light_mode", "theme", "mode"}:
            return "user", relation_hit, "prefers"
        return "user", relation_hit, f"favorite_{slot}"

    target_map = {
        "live": "lives_in", "lives": "lives_in", "location": "lives_in", "city": "lives_in",
        "job": "job", "work": "job", "role": "job",
        "salary": "salary", "pay": "salary", "income": "salary",
        "own": "owns", "owns": "owns", "ownership": "owns",
        "prefer": "prefers", "preference": "prefers", "theme": "prefers",
        "pet": "has_pet", "dog": "has_pet", "cat": "has_pet",
    }
    target_hit = next((v for k, v in target_map.items() if re.search(rf"\b{k}\b", q)), None)
    if relation_hit and target_hit:
        return "user", relation_hit, target_hit
    return None

### 9.1 Multi-hop demo fixture

Adds a relation edge (`user → reports_to → manager_alex`) plus a second, independent revision chain on the related entity (`manager_alex` moved Berlin → Tokyo). Ingested through the same `ingest_facts_with_graph` used for the rest of the demo — no new ingestion path required.

In [ ]:
multi_hop_facts = [
    make_fact(
        "rel_reports_001", 10, 15,
        "user", "reports_to", "manager_alex",
        fact_type="relation",
        text="I report to Alex."
    ),
    make_fact(
        "mgr_loc_001", 11, 16,
        "manager_alex", "lives_in", "Berlin",
        text="Alex lives in Berlin."
    ),
    make_fact(
        "mgr_loc_002", 12, 40,
        "manager_alex", "lives_in", "Tokyo",
        supersedes="mgr_loc_001",
        text="Alex relocated to Tokyo."
    ),
]

multi_hop_ids = ingest_facts_with_graph(multi_hop_facts, DATABASE)
wait_for_indexing(multi_hop_ids, DATABASE)
print("Multi-hop fixture ingested and indexed.")

Ingested 3 memories with BYOG graph.
Indexing completed (graph ready).
Multi-hop fixture ingested and indexed.


### 9.2 Extended query path — single-hop or multi-hop, decided per query

Tries a two-hop match first; falls back to the existing single-hop path unchanged. Everything downstream (HydraDB call, `graph_context=True`, chunk-to-fact conversion) is reused as-is.

In [ ]:
def answer_memory_query_v2(
    query: str,
    database: str,
    top_k: int = 30,
) -> Dict[str, Any]:
    """
    Same retrieval path as answer_memory_query, but checks for a multi-hop
    target first. Falls back to the original single-hop behaviour untouched.
    """
    multi_hop_target = extract_multi_hop_target(query)

    reset_chunk_drop_count()
    result = hydra_query(
        client,
        database=database,
        query=query,
        type="memory",
        query_by="hybrid",
        mode="thinking",
        max_results=top_k,
        graph_context=True,
        # temporal kwargs intentionally omitted (see Section 6).
    )
    chunks = obj_get(result.data, "chunks", []) or []
    graph_ctx = obj_get(result.data, "graph_context", {}) or {}

    candidate_facts = []
    for ch in chunks:
        fact = hydra_chunk_to_fact(ch)
        if fact:
            candidate_facts.append(fact)
    if get_chunk_drop_count():
        print(f"  [answer] WARNING: {get_chunk_drop_count()} retrieved chunk(s) "
              f"could not be parsed as facts -- see drop log above.")

    if multi_hop_target:
        subject, relation_predicate, target_predicate = multi_hop_target
        resolution = resolve_multi_hop(candidate_facts, subject, relation_predicate, target_predicate)
        target_label = f"{subject}::{relation_predicate}->{target_predicate}"
        hops_payload = [
            {
                "state": h["state"],
                "answer": h["fact"].object_value if h.get("fact") else None,
                "reason": h["reason"],
            }
            for h in resolution["hops"]
        ]
    else:
        subject, predicate = extract_target_from_query(query)
        resolution = resolve_revision_state(candidate_facts, subject, predicate)
        target_label = f"{subject}::{predicate}"
        hops_payload = None

    answer = None
    if resolution["state"] == MemoryState.ANSWERABLE.value and resolution["fact"]:
        answer = resolution["fact"].object_value

    return {
        "query": query,
        "target": target_label,
        "multi_hop": bool(multi_hop_target),
        "related_entity": resolution.get("related_entity"),
        "state": resolution["state"],
        "answer": answer,
        "reason": resolution["reason"],
        "hops": hops_payload,
        "revision_chain": [
            {"fact_id": f.fact_id, "value": f.object_value, "ts": f.timestamp.isoformat()}
            for f in resolution.get("chain", [])
        ] if not multi_hop_target else [],
        "retrieved_candidates": len(candidate_facts),
        "graph_context": {
            "query_paths": obj_get(graph_ctx, "query_paths", []),
            "chunk_relations": obj_get(graph_ctx, "chunk_relations", []),
        },
    }


def show_v2_answer(result: Dict[str, Any]) -> None:
    print(f"Query      : {result['query']}")
    print(f"Target     : {result['target']}  (multi-hop: {result['multi_hop']})")
    print(f"State      : {result['state']}")
    print(f"Answer     : {result['answer']}")
    print(f"Reason     : {result['reason']}")
    if result["multi_hop"] and result["hops"]:
        print("\nHop-by-hop evidence:")
        for i, h in enumerate(result["hops"], start=1):
            print(f"  hop {i}: state={h['state']}  answer={h['answer']}")
    print(f"\nCandidates retrieved: {result['retrieved_candidates']}")

### 9.3 DEMO 4 — Live multi-hop query

This is the fourth demo case for the video: a question the notebook's original single-hop resolver structurally cannot answer, and that a vector-similarity search cannot answer either, because the phrase "manager's city" never appears verbatim anywhere in the corpus.

In [ ]:
print("=" * 60)
print("DEMO 4 — Multi-hop relationship traversal")
print("=" * 60)
result4 = answer_memory_query_v2("Where does the user's manager live?", DATABASE)
show_v2_answer(result4)
print("\nExpected: ANSWERABLE -> Tokyo (via user -> reports_to -> manager_alex -> lives_in)")
print("PASS ✅" if (result4["state"] == "ANSWERABLE" and result4["answer"] == "Tokyo") else "FAIL ❌ — check indexing finished / graph_context payload shape")

DEMO 4 — Multi-hop relationship traversal
Query      : Where does the user's manager live?
Target     : user::reports_to->lives_in  (multi-hop: True)
State      : ANSWERABLE
Answer     : Tokyo
Reason     : Hop 1 resolved 'user.reports_to' -> 'manager_alex'; Hop 2 resolved 'manager_alex.lives_in': Resolved from the terminal fact in the revision chain.

Hop-by-hop evidence:
  hop 1: state=ANSWERABLE  answer=manager_alex
  hop 2: state=ANSWERABLE  answer=Tokyo

Candidates retrieved: 13

Expected: ANSWERABLE -> Tokyo (via user -> reports_to -> manager_alex -> lives_in)
PASS ✅


## 10. Vector-only baseline comparison

Directly targets "a use case that is hard to pull off with vector or relational approaches" from the judging rubric — up to now that claim was asserted, not shown.

This baseline simulates a standard RAG/vector-store memory layer: embed every fact's text, embed the query, return the single most-similar chunk's content. No temporal reasoning, no supersession awareness, no abstention concept, no multi-hop traversal — because a vector index has no structural way to represent any of those.

We use TF-IDF cosine similarity as a fast, dependency-light stand-in for a real embedding model. This is a fair proxy, not a strawman: the failure mode below is structural (no graph/temporal reasoning), not a quality-of-embedding issue — a strong embedding model would still rank "I live in New York" / "I moved to San Francisco" / "I relocated to London" as similarly relevant to "Where does the user live?", because nothing in the *text itself* marks any one of them as current. That's exactly the gap HydraMem's `SUPERSEDES` graph edges are built to close.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def vector_only_answer(query: str, corpus: List[MemoryFact]) -> Dict[str, Any]:
    """Naive vector-store baseline: pure similarity, top-1 wins, no reasoning."""
    texts = [f.source_text or f"{f.subject} {f.predicate} {f.object_value}" for f in corpus]
    if not texts:
        return {"answer": None, "top_fact": None, "similarity": None}

    vec = TfidfVectorizer().fit(texts + [query])
    doc_vecs = vec.transform(texts)
    q_vec = vec.transform([query])
    sims = cosine_similarity(q_vec, doc_vecs)[0]

    best_idx = sims.argmax()
    best_fact = corpus[best_idx]
    # A vector store has no abstention mechanism -- it always returns its
    # best match, even when similarity is low or the fact is unrelated.
    return {"answer": best_fact.object_value, "top_fact": best_fact, "similarity": float(sims[best_idx])}


def run_baseline_comparison(benchmark_facts: List[MemoryFact]) -> pd.DataFrame:
    conflict_corpus = [
        make_fact("bench_conflict_a", 100, 50, "user", "lives_in", "Boston", text="I live in Boston now."),
        make_fact("bench_conflict_b", 101, 50, "user", "lives_in", "Cambridge", text="I actually live in Cambridge now."),
    ]

    cases = [
        {"name": "current location (revision)", "query": "Where does the user live?",
         "expected": "Boston",       # hand-authored ground truth (seed=7 generator)
         "corpus": benchmark_facts, "mode": "single", "predicate": "lives_in"},
        {"name": "current job (overwrite)", "query": "What is the user's job?",
         "expected": "researcher",   # hand-authored ground truth
         "corpus": benchmark_facts, "mode": "single", "predicate": "job"},
        {"name": "static preference", "query": "What does the user prefer?",
         "expected": "dark mode", "corpus": benchmark_facts, "mode": "single", "predicate": "prefers"},
        {"name": "missing fact (abstention)", "query": "What is the user's favorite pet?",
         "expected": None, "corpus": benchmark_facts, "mode": "single", "predicate": "unknown_property"},
        {"name": "concurrent conflict", "query": "Where does the user live?",
         "expected": None, "corpus": conflict_corpus, "mode": "conflict", "predicate": "lives_in"},
        {"name": "multi-hop (manager's location)", "query": "Where does the user's manager live?",
         "expected": "Tokyo", "corpus": multi_hop_facts, "mode": "multihop", "predicate": None},
    ]

    rows = []
    for case in cases:
        if case["mode"] == "multihop":
            target = extract_multi_hop_target(case["query"])
            if target is None:
                hm = {"state": MemoryState.NO_EVIDENCE.value, "fact": None,
                      "reason": "multi-hop target not recognized"}
            else:
                hm = resolve_multi_hop(case["corpus"], *target)
        else:
            hm = resolve_revision_state(case["corpus"], "user", case["predicate"])
        hm_answer = hm["fact"].object_value if hm.get("fact") else None

        if case["mode"] == "conflict":
            hm_pass = hm["state"] == MemoryState.CONFLICTING.value
        elif case["expected"] is None:
            hm_pass = hm["state"] == MemoryState.NO_EVIDENCE.value
        else:
            hm_pass = hm_answer == case["expected"]

        vec = vector_only_answer(case["query"], case["corpus"])
        vec_answer = vec["answer"]
        if case["mode"] == "conflict":
            # A vector store has no mechanism to flag ambiguity -- by design it
            # cannot pass this case. This is a documented judgment, not a measurement.
            vec_pass = False
        elif case["expected"] is None:
            vec_pass = vec_answer is None  # no mechanism to abstain
        else:
            vec_pass = vec_answer == case["expected"]

        rows.append({
            "case": case["name"],
            "expected": case["expected"],
            "hydramem_answer": hm_answer,
            "hydramem_state": hm["state"],
            "hydramem_correct": hm_pass,
            "vector_baseline_answer": vec_answer,
            "vector_baseline_correct": vec_pass,
        })

    return pd.DataFrame(rows)


comparison_df = run_baseline_comparison(benchmark_facts)
display(comparison_df)

print(f"\nHydraMem accuracy       : {comparison_df['hydramem_correct'].mean():.1%} "
      f"({comparison_df['hydramem_correct'].sum()}/{len(comparison_df)})")
print(f"Vector-baseline accuracy: {comparison_df['vector_baseline_correct'].mean():.1%} "
      f"({comparison_df['vector_baseline_correct'].sum()}/{len(comparison_df)})")

,case,expected,hydramem_answer,hydramem_state,hydramem_correct,vector_baseline_answer,vector_baseline_correct
0,current location (revision),Boston,Boston,ANSWERABLE,True,London,False
1,current job (overwrite),researcher,researcher,ANSWERABLE,True,researcher,True
2,static preference,dark mode,dark mode,ANSWERABLE,True,dark mode,True
3,missing fact (abstention),None,None,NO_EVIDENCE,True,researcher,False
4,concurrent conflict,None,None,CONFLICTING,True,Boston,False
5,multi-hop (manager's location),Tokyo,Tokyo,ANSWERABLE,True,manager_alex,False



HydraMem accuracy       : 100.0% (6/6)
Vector-baseline accuracy: 33.3% (2/6)


**Reading the result:** the baseline gets static, never-revised facts right (job/preference lookups where only one fact ever existed) — that's expected and honest. A vector store isn't *bad*, it's just blind to structure. It fails specifically on the four things that require structure: which revision is current, when to abstain, when to flag conflict, and anything requiring a hop across entities. Those four are exactly what the Track 3 brief calls out as the hard part.

## 11. Rule-based fact extraction from dialogue (no LLM)

Real LongMemEval / BEAM data arrives as free-form user/assistant turns, not pre-typed facts. This section adds a **deterministic, pattern-based extractor** that turns dialogue text into `MemoryFact`s.

Goals:
- Cover the high-frequency predicates needed for the benchmarks (`lives_in`, `job`, `prefers`)
- Detect revision language ("I moved", "now I live", "I changed jobs") so we can set `supersedes_fact_id`
- Stay fully rule-based — no LLM call
- Feed the extracted facts straight into the existing revision resolver

**v2.2 hardening:**
- City capture is a case-**sensitive** capitalized-run matcher: `"I moved to San Francisco last week."` now yields `'San Francisco'`, not `'San Francisco last week'`
- Preference capture stops at trailing `for ...` / `because ...` clauses: `'dark mode'`, not `'dark mode for my apps'`
- `link_supersession` no longer chains same-timestamp facts — simultaneous contradictions stay terminal so the resolver can flag `CONFLICTING` through the live extraction path

In [ ]:
import re
from typing import List, Optional, Dict, Tuple
from datetime import datetime, timezone

# ---------------------------------------------------------------------------
# Pattern tables (ordered: more specific first)
# Each entry: (compiled regex, predicate, fact_type, confidence)
# Capture group 1 = object_value
# ---------------------------------------------------------------------------

# Case-SENSITIVE capitalized-run city matcher. re.I is deliberately NOT used on
# these patterns: with re.I, [A-Z] would also match lowercase and words like
# "last week" would bleed into the captured city ("San Francisco last week").
_CITY = r"((?:[A-Z][A-Za-z\-]+)(?:\s+[A-Z][A-Za-z\-]+)*)"

_LOCATION_PATTERNS = [
    # "I moved to London" / "I just moved to San Francisco last week"
    (re.compile(rf"\b[Ii]\s+(?:just\s+)?moved\s+to\s+{_CITY}"), "lives_in", "volatile", 0.95),
    # "I live in New York" / "I now live in Boston"
    (re.compile(rf"\b[Ii]\s+(?:now\s+)?live\s+in\s+{_CITY}"), "lives_in", "volatile", 0.95),
    # "I relocated to Tokyo" / "I'm based in Berlin" / "I reside in Chicago"
    (re.compile(rf"\b[Ii]\s+relocated\s+to\s+{_CITY}"), "lives_in", "volatile", 0.9),
    (re.compile(rf"\b[Ii](?:'m|\s+am)\s+based\s+in\s+{_CITY}"), "lives_in", "volatile", 0.9),
    (re.compile(rf"\b[Ii]\s+reside\s+in\s+{_CITY}"), "lives_in", "volatile", 0.85),
]

# Job titles may be lowercase; stop at sentence end or a new clause.
_JOB_STOP = r"(?:\s+at\s|\s+for\s|\s+and\s|\s+but\s|\.|,|!|$)"
_JOB_PATTERNS = [
    (re.compile(rf"\b[Ii]\s+(?:now\s+)?work\s+as\s+(?:a\s+|an\s+)?([a-zA-Z][a-zA-Z\s\-]*?){_JOB_STOP}", re.I), "job", "volatile", 0.9),
    (re.compile(rf"\b[Ii]\s+(?:just\s+)?(?:got|started|took)\s+(?:a\s+|an\s+)?(?:new\s+)?job\s+as\s+(?:a\s+|an\s+)?([a-zA-Z][a-zA-Z\s\-]*?){_JOB_STOP}", re.I), "job", "volatile", 0.9),
    (re.compile(r"\b[Ii](?:'m|\s+am)\s+(?:a\s+|an\s+)?([a-zA-Z\s\-]*(?:engineer|designer|manager|researcher|developer|analyst|scientist))\b", re.I), "job", "volatile", 0.85),
    (re.compile(rf"\b[Mm]y\s+job\s+is\s+(?:a\s+|an\s+)?([a-zA-Z][a-zA-Z\s\-]*?){_JOB_STOP}", re.I), "job", "volatile", 0.85),
]

# Preferences: stop before trailing "for ..."/"because ..." clauses.
_PREF_STOP = r"(?:\s+for\s|\s+because\s|\s+when\s|\s+since\s|\.|,|!|$)"
_PREF_PATTERNS = [
    (re.compile(r"\b[Ii]\s+(?:really\s+)?prefer\s+(dark\s+mode|light\s+mode)\b"), "prefers", "static", 0.95),
    (re.compile(rf"\b[Ii]\s+(?:really\s+)?prefer\s+([a-zA-Z][a-zA-Z\s\-]*?){_PREF_STOP}"), "prefers", "static", 0.9),
    (re.compile(r"\b[Ii]\s+like\s+([a-zA-Z\s\-]*?dark\s+mode[a-zA-Z\s\-]*?)(?:\.|,|!|$)"), "prefers", "static", 0.9),
]

ALL_PATTERNS = _LOCATION_PATTERNS + _JOB_PATTERNS + _PREF_PATTERNS

# Phrases that strongly signal a revision of a previous value of the same property
REVISION_CUES = re.compile(
    r"\b(moved|relocated|now live|now work|changed|switched|updated|no longer|instead)\b",
    re.I,
)


def clean_value(raw: str) -> str:
    v = raw.strip().rstrip(".!,")
    # collapse internal whitespace
    v = re.sub(r"\s+", " ", v)
    return v


def extract_facts_from_text(
    text: str,
    *,
    session_id: str,
    timestamp: datetime,
    subject: str = "user",
    fact_id_prefix: str = "auto",
    counter_start: int = 0,
) -> List[MemoryFact]:
    """
    Rule-based extraction of MemoryFacts from a single dialogue turn.
    Returns zero or more facts. Does NOT set supersedes_fact_id yet
    (that is done by the session-level linker below).
    """
    facts: List[MemoryFact] = []
    seen_preds = set()
    idx = counter_start

    # Open-vocabulary fallbacks for common non-core attributes.
    # These supplement (not replace) the stable core regex patterns.
    open_patterns = [
        (re.compile(r"\b(?:my\s+)?(?:favorite|favourite)\s+([a-z][a-z0-9\-]*(?:\s+[a-z][a-z0-9\-]*)?)\s+is\s+([^.!?,]+)", re.I), "favorite", "static", 0.88),
        (re.compile(r"\b(?:my\s+)?pet\s+(?:is|named)\s+([^.!?,]+)", re.I), "has_pet", "static", 0.92),
        (re.compile(r"\b(?:i|I)\s+(?:have|got)\s+(?:a|an)\s+(?:dog|cat|pet)(?:\s+(?:named|called))?\s+([^.!?,]+)", re.I), "has_pet", "static", 0.92),
        (re.compile(r"\b(?:my\s+)?(?:salary|pay|compensation|income)\s+(?:is|=)\s+([^.!?,]+)", re.I), "salary", "static", 0.9),
        (re.compile(r"\b(?:i|I)\s+(?:make|earn)\s+([^.!?,]+)", re.I), "salary", "static", 0.82),
        (re.compile(r"\b(?:i|I)\s+(?:own)\s+([^.!?,]+)", re.I), "owns", "static", 0.88),
    ]

    for pattern, predicate, fact_type, conf in open_patterns:
        if predicate == "favorite":
            m = pattern.search(text)
            if not m:
                continue
            slot = re.sub(r"[\s\-]+", " ", m.group(1).strip().lower())
            predicate = "has_pet" if slot in {"pet", "dog", "cat", "puppy", "kitten"} else f"favorite_{slot}"
            value = clean_value(m.group(2))
        else:
            if predicate in seen_preds:
                continue
            m = pattern.search(text)
            if not m:
                continue
            value = clean_value(m.group(1))

        if len(value) < 2 or len(value) > 100 or predicate in seen_preds:
            continue
        local_conf = min(1.0, conf + 0.05) if REVISION_CUES.search(text) else conf
        fact_id = f"{fact_id_prefix}_{predicate}_{idx:03d}"
        idx += 1
        seen_preds.add(predicate)
        facts.append(MemoryFact(
            fact_id=fact_id,
            subject=subject,
            predicate=predicate,
            object_value=value,
            timestamp=timestamp,
            session_id=session_id,
            fact_type=fact_type,
            supersedes_fact_id=None,
            source_text=text.strip(),
            confidence=local_conf,
        ))

    for pattern, predicate, fact_type, conf in ALL_PATTERNS:
        if predicate in seen_preds:
            continue  # one fact per predicate per turn
        m = pattern.search(text)
        if not m:
            continue
        value = clean_value(m.group(1))
        if len(value) < 2 or len(value) > 60:
            continue

        # Boost confidence slightly if revision language is present
        local_conf = conf
        if REVISION_CUES.search(text):
            local_conf = min(1.0, conf + 0.05)

        fact_id = f"{fact_id_prefix}_{predicate}_{idx:03d}"
        idx += 1
        seen_preds.add(predicate)

        facts.append(MemoryFact(
            fact_id=fact_id,
            subject=subject,
            predicate=predicate,
            object_value=value,
            timestamp=timestamp,
            session_id=session_id,
            fact_type=fact_type,
            supersedes_fact_id=None,  # linked later
            source_text=text.strip(),
            confidence=local_conf,
        ))

    return facts


def link_supersession(facts: List[MemoryFact]) -> List[MemoryFact]:
    """
    Chronologically link revision chains. Same-timestamp facts for the same
    (subject, predicate) are intentionally NOT linked: leaving both terminal
    lets resolve_revision_state flag CONFLICTING when their values differ,
    instead of silently letting insertion order decide the truth.
    """
    # sort stably by timestamp then original order
    ordered = sorted(enumerate(facts), key=lambda x: (x[1].timestamp, x[0]))
    latest: Dict[Tuple[str, str], Tuple[str, datetime]] = {}  # (subject, predicate) -> (fact_id, ts)
    result: List[MemoryFact] = []

    for _, f in ordered:
        key = (f.subject, f.predicate)
        prev = latest.get(key)
        if prev and prev[0] != f.fact_id and f.timestamp > prev[1]:
            # create a new model instance with supersedes set
            f = f.model_copy(update={"supersedes_fact_id": prev[0]})
        latest[key] = (f.fact_id, f.timestamp)
        result.append(f)

    return result


def extract_facts_from_sessions(
    sessions: List[Dict],
    *,
    default_subject: str = "user",
) -> List[MemoryFact]:
    """
    sessions: list of dicts with keys:
        - session_id: str
        - timestamp: datetime
        - turns: list of {"role": "user"|"assistant", "content": str}

    Only user turns are extracted by default (LongMemEval evidence is mostly user-side).
    """
    all_facts: List[MemoryFact] = []
    counter = 0

    for sess in sessions:
        sid = sess["session_id"]
        ts = sess["timestamp"]
        for turn in sess.get("turns", []):
            if turn.get("role") != "user":
                continue
            text = turn.get("content") or ""
            extracted = extract_facts_from_text(
                text,
                session_id=sid,
                timestamp=ts,
                subject=default_subject,
                fact_id_prefix="auto",
                counter_start=counter,
            )
            counter += len(extracted)
            all_facts.extend(extracted)

    return link_supersession(all_facts)


# ---------------------------------------------------------------------------
# Demo: synthetic LongMemEval-style dialogue -> MemoryFacts -> resolver
# ---------------------------------------------------------------------------

demo_sessions = [
    {
        "session_id": "sess_001",
        "timestamp": datetime(2026, 8, 1, 10, 0, tzinfo=timezone.utc),
        "turns": [
            {"role": "user", "content": "Hi, I live in New York."},
            {"role": "assistant", "content": "Got it, New York."},
        ],
    },
    {
        "session_id": "sess_002",
        "timestamp": datetime(2026, 8, 2, 10, 0, tzinfo=timezone.utc),
        "turns": [
            {"role": "user", "content": "I prefer dark mode for my apps."},
        ],
    },
    {
        "session_id": "sess_003",
        "timestamp": datetime(2026, 8, 5, 10, 0, tzinfo=timezone.utc),
        "turns": [
            {"role": "user", "content": "I moved to San Francisco last week."},
        ],
    },
    {
        "session_id": "sess_004",
        "timestamp": datetime(2026, 8, 8, 10, 0, tzinfo=timezone.utc),
        "turns": [
            {"role": "user", "content": "I work as a software engineer."},
        ],
    },
    {
        "session_id": "sess_005",
        "timestamp": datetime(2026, 8, 11, 10, 0, tzinfo=timezone.utc),
        "turns": [
            {"role": "user", "content": "I relocated to London."},
        ],
    },
]

extracted_facts = extract_facts_from_sessions(demo_sessions)

print("Extracted facts (rule-based, no LLM):")
for f in extracted_facts:
    sup = f"  supersedes={f.supersedes_fact_id}" if f.supersedes_fact_id else ""
    print(f"  {f.fact_id}: ({f.subject}, {f.predicate}) -> {f.object_value!r} @ {f.timestamp.date()}{sup}")

print("\n=== Resolution on extracted facts ===")
loc = resolve_revision_state(extracted_facts, "user", "lives_in")
pretty_resolution(loc)
print("Expected: ANSWERABLE -> London")
print("PASS ✅" if (loc["state"] == MemoryState.ANSWERABLE.value and loc["fact"].object_value == "London") else "FAIL ❌")

job = resolve_revision_state(extracted_facts, "user", "job")
print("\nJob:", job["state"], "->", job["fact"].object_value if job["fact"] else None)

pref = resolve_revision_state(extracted_facts, "user", "prefers")
print("Prefers:", pref["state"], "->", pref["fact"].object_value if pref["fact"] else None)

missing = resolve_revision_state(extracted_facts, "user", "favorite_pet")
print("Missing (favorite_pet):", missing["state"])

Extracted facts (rule-based, no LLM):
  auto_lives_in_000: (user, lives_in) -> 'New York' @ 2026-08-01
  auto_prefers_001: (user, prefers) -> 'dark mode' @ 2026-08-02
  auto_lives_in_002: (user, lives_in) -> 'San Francisco' @ 2026-08-05  supersedes=auto_lives_in_000
  auto_job_003: (user, job) -> 'software engineer' @ 2026-08-08
  auto_lives_in_004: (user, lives_in) -> 'London' @ 2026-08-11  supersedes=auto_lives_in_002

=== Resolution on extracted facts ===
State : ANSWERABLE
Reason: Resolved from the terminal fact in the revision chain.
Answer: London
Fact ID: auto_lives_in_004
Timestamp: 2026-08-11T10:00:00+00:00
Revision chain:
  auto_lives_in_000: New York @ 2026-08-01
  auto_lives_in_002: San Francisco @ 2026-08-05
  auto_lives_in_004: London @ 2026-08-11
Expected: ANSWERABLE -> London
PASS ✅

Job: ANSWERABLE -> software engineer
Prefers: ANSWERABLE -> dark mode
Missing (favorite_pet): NO_EVIDENCE


In [ ]:
# Proof that CONFLICTING is now reachable through the LIVE extraction path.
# Before v2.2 the old linker silently chained same-timestamp contradictions by
# insertion order, so only hand-built fixtures could produce CONFLICTING.
same_ts_sessions = [
    {"session_id": "sess_900", "timestamp": datetime(2026, 8, 12, 10, 0, tzinfo=timezone.utc),
     "turns": [{"role": "user", "content": "I live in Boston."}]},
    {"session_id": "sess_901", "timestamp": datetime(2026, 8, 12, 10, 0, tzinfo=timezone.utc),
     "turns": [{"role": "user", "content": "I now live in Cambridge."}]},
]
same_ts_facts = extract_facts_from_sessions(same_ts_sessions)
r = resolve_revision_state(same_ts_facts, "user", "lives_in")
print(f"State: {r['state']} -- {r['reason']}")
assert r["state"] == MemoryState.CONFLICTING.value, "same-timestamp conflict must be reachable via live extraction"
print("PASS ✅ same-timestamp conflict now surfaces through the extraction pipeline")

State: CONFLICTING -- Multiple current contradictory values exist at the same timestamp.
PASS ✅ same-timestamp conflict now surfaces through the extraction pipeline


## 11b. Hybrid extraction — Claude Structured Outputs + our deterministic resolver

This is an **additional** extraction path alongside the rule-based extractor above (Section 11), not a replacement. Both feed the exact same deterministic pipeline downstream: `link_supersession` -> `resolve_revision_state` -> `ANSWERABLE / CONFLICTING / NO_EVIDENCE`.

**Division of labor:**
- **Claude** does one job: parse messy natural language into structured candidate facts. It uses **Structured Outputs** via `output_config.format={"type": "json_schema", ...}`. The API constrains the response to the declared JSON Schema, while `predicate` remains an open string so properties such as `favorite_color`, `has_pet`, `salary`, and `owns` are allowed.
- Our **existing deterministic resolver** does everything after that. Claude never decides what's true — it only proposes structured candidates.

**Why Structured Outputs:** the API guarantees schema-compliant JSON for normal successful responses, so we don't depend on prompt formatting or markdown-fence stripping. We still parse the returned text block with `json.loads()` because the JSON output is delivered there.

**Why this is the direct mem0 comparison point:** the LLM is limited to extraction. The truth-decision remains deterministic in HydraMem: supersession linking, revision resolution, conflict detection, and abstention are all handled by application logic.

**Fails safe:** API errors, refusals, truncation, missing structured output, malformed JSON, or empty/invalid facts fall back to the rule-based extractor from Section 11. Ingestion never stalls on a bad LLM call.

In [ ]:
CORE_PREDICATES = [
    "lives_in",
    "job",
    "prefers",
    "reports_to",
    "collaborates_with",
]
KNOWN_PREDICATES = CORE_PREDICATES


def normalize_predicate(predicate: Any) -> str:
    """Canonicalize an extracted predicate without imposing a closed enum."""
    p = str(predicate or "").strip().lower()
    p = re.sub(r"[\s\-]+", " ", p)
    p = re.sub(r"[^a-z0-9 ]+", " ", p)
    p = re.sub(r" +", " ", p).strip(" ")
    return p


FACT_EXTRACTION_SCHEMA = {
    "type": "object",
    "properties": {
        "facts": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "subject": {"type": "string"},
                    "predicate": {
                        "type": "string",
                        "description": "Short snake_case property/relation. Prefer core predicates when applicable; otherwise create a concise descriptive predicate."
                    },
                    "object_value": {"type": "string"},
                    "fact_type": {"type": "string", "enum": ["volatile", "static", "episodic"]},
                    "confidence": {"type": "number", "description": "Confidence from 0 to 1."}
                },
                "required": ["subject", "predicate", "object_value", "fact_type", "confidence"],
                "additionalProperties": False
            }
        }
    },
    "required": ["facts"],
    "additionalProperties": False
}


def load_llm_api_key() -> Optional[str]:
    try:
        from google.colab import userdata
        key = userdata.get("ANTHROPIC_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return os.getenv("ANTHROPIC_API_KEY")


def _call_llm_extractor_structured(text: str, llm_client_obj) -> List[dict]:
    """
    Use Claude Structured Outputs via output_config.format=json_schema.

    Claude returns schema-valid JSON in a text content block. We parse only that
    validated JSON and keep the deterministic resolver downstream. Any refusal,
    truncation, malformed response, or API error triggers the caller's fallback.
    """
    response = llm_client_obj.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": (
                f"Extract facts from this message. Prefer core predicates {CORE_PREDICATES}; "
                f"otherwise create a short snake_case predicate (for example favorite_color, has_pet, salary, owns). "
                f"Return an empty facts list when no fact is present. Never invent facts not present in the text. "
                f"Return only data matching the provided schema.\n\nMessage: {text}"
            ),
        }],
        output_config={
            "format": {
                "type": "json_schema",
                "schema": FACT_EXTRACTION_SCHEMA,
            }
        },
    )

    text_blocks = [
        block.text
        for block in response.content
        if getattr(block, "type", None) == "text" and getattr(block, "text", None)
    ]
    if not text_blocks:
        stop_reason = getattr(response, "stop_reason", None)
        raise ValueError(f"No structured text block returned (stop_reason={stop_reason!r})")

    payload = json.loads(text_blocks[0])
    facts = payload.get("facts")
    if not isinstance(facts, list):
        raise ValueError("Structured output missing 'facts' list")
    return facts


def extract_facts_from_text_llm(
    text: str,
    *,
    session_id: str,
    timestamp: datetime,
    subject: str = "user",
    fact_id_prefix: str = "llm",
    counter_start: int = 0,
    llm_client=None,
) -> List[MemoryFact]:
    """
    Structured-output LLM extraction with a hard fallback to extract_facts_from_text
    (Section 11). Never raises: API errors, refusals/truncation, malformed structured
    JSON, or empty/invalid results fall back to the rule-based extractor so ingestion
    never stalls on a bad LLM call.
    """
    if llm_client is None:
        return extract_facts_from_text(
            text, session_id=session_id, timestamp=timestamp, subject=subject,
            fact_id_prefix="auto", counter_start=counter_start,
        )

    try:
        parsed = _call_llm_extractor_structured(text, llm_client)
    except Exception as e:
        print(f"  [extraction] LLM call failed ({e}); falling back to rule-based extractor")
        return extract_facts_from_text(
            text, session_id=session_id, timestamp=timestamp, subject=subject,
            fact_id_prefix="auto", counter_start=counter_start,
        )

    facts: List[MemoryFact] = []
    idx = counter_start
    for item in parsed:
        try:
            pred = normalize_predicate(item.get("predicate"))
            val = (item.get("object_value") or "").strip()
            if not pred or len(pred) > 64 or not val:
                continue
            fact_id = f"{fact_id_prefix}_{pred}_{idx:03d}"
            idx += 1
            facts.append(MemoryFact(
                fact_id=fact_id,
                subject=item.get("subject") or subject,
                predicate=pred,
                object_value=val,
                timestamp=timestamp,
                session_id=session_id,
                fact_type=item.get("fact_type", "volatile"),
                supersedes_fact_id=None,
                source_text=text.strip(),
                confidence=float(item.get("confidence", 0.75)),
            ))
        except Exception:
            continue

    if not facts:
        return extract_facts_from_text(
            text, session_id=session_id, timestamp=timestamp, subject=subject,
            fact_id_prefix="auto", counter_start=counter_start,
        )
    return facts


def extract_facts_from_sessions_hybrid(
    sessions: List[Dict],
    *,
    default_subject: str = "user",
    llm_client=None,
) -> List[MemoryFact]:
    """
    Same contract as extract_facts_from_sessions (Section 11), but routes each turn
    through Claude Structured Outputs with automatic fallback instead of the
    rule-based extractor directly. Pass llm_client=None to reproduce the original
    rule-based behavior exactly.
    """
    all_facts: List[MemoryFact] = []
    counter = 0

    for sess in sessions:
        sid = sess["session_id"]
        ts = sess["timestamp"]
        for turn in sess.get("turns", []):
            if turn.get("role") != "user":
                continue
            text = turn.get("content") or ""
            extracted = extract_facts_from_text_llm(
                text,
                session_id=sid,
                timestamp=ts,
                subject=default_subject,
                counter_start=counter,
                llm_client=llm_client,
            )
            counter += len(extracted)
            all_facts.extend(extracted)

    return link_supersession(all_facts)

print("Hybrid extractor (Claude Structured Outputs) defined: extract_facts_from_text_llm, extract_facts_from_sessions_hybrid")

Hybrid extractor (Claude Structured Outputs) defined: extract_facts_from_text_llm, extract_facts_from_sessions_hybrid


In [ ]:
# ---------------------------------------------------------------------------
# Demo: messy dialogue that breaks the regex extractor (Section 11),
# handled correctly once routed through the structured-output LLM path.
# Also shows the structured fact feeding directly into the same graph-entity
# shape used by BYOG ingestion (Section 5) -- no adapter layer needed.
# ---------------------------------------------------------------------------
messy_sessions = [
    {
        "session_id": "sess_101",
        "timestamp": datetime(2026, 8, 1, 10, 0, tzinfo=timezone.utc),
        "turns": [{"role": "user", "content": "hey so quick update, I'm in New York these days"}],
    },
    {
        "session_id": "sess_102",
        "timestamp": datetime(2026, 8, 5, 10, 0, tzinfo=timezone.utc),
        "turns": [{"role": "user", "content": "yeah I ended up moving, London now, work's been crazy"}],
    },
]

# import anthropic
# llm_client = anthropic.Anthropic(api_key=load_llm_api_key())
llm_client = None  # set to a real client to run this against the live API

print("=== A: Rule-based extractor (Section 11) on messy text ===")
rule_only = extract_facts_from_sessions(messy_sessions)
for f in rule_only:
    print(f"  {f.fact_id}: ({f.subject}, {f.predicate}) -> {f.object_value!r}")
if not rule_only:
    print("  (nothing extracted -- this is the gap the LLM path closes)")

print("\n=== B: Hybrid extractor, llm_client=None (reproduces A exactly -- same fallback path) ===")
hybrid_no_client = extract_facts_from_sessions_hybrid(messy_sessions, llm_client=None)
print(f"  {len(hybrid_no_client)} facts (should match A's count)")
print("  PASS ✅" if len(hybrid_no_client) == len(rule_only) else "  FAIL ❌")

print("\n=== C: Hybrid extractor with a real LLM client (set llm_client above to run live) ===")
print("  Expected once wired up: lives_in -> London (correctly parsed from messy phrasing),")
print("  superseding the earlier New York fact -- schema-guaranteed, no parsing fallback needed.")
if llm_client is not None:
    hybrid_live = extract_facts_from_sessions_hybrid(messy_sessions, llm_client=llm_client)
    for f in hybrid_live:
        sup = f"  supersedes={f.supersedes_fact_id}" if f.supersedes_fact_id else ""
        print(f"  {f.fact_id}: ({f.subject}, {f.predicate}) -> {f.object_value!r} conf={f.confidence}{sup}")
    loc = resolve_revision_state(hybrid_live, "user", "lives_in")
    pretty_resolution(loc)

    print("\n=== D: structured fact -> graph entities/relations, straight through (Section 2) ===")
    if hybrid_live:
        graph_fragment = hybrid_live[-1].to_graph_entities_and_relations()
        print(graph_fragment)

=== A: Rule-based extractor (Section 11) on messy text ===
  (nothing extracted -- this is the gap the LLM path closes)

=== B: Hybrid extractor, llm_client=None (reproduces A exactly -- same fallback path) ===
  0 facts (should match A's count)
  PASS ✅

=== C: Hybrid extractor with a real LLM client (set llm_client above to run live) ===
  Expected once wired up: lives_in -> London (correctly parsed from messy phrasing),
  superseding the earlier New York fact -- schema-guaranteed, no parsing fallback needed.


## 11c. Read-latency instrumentation

Measurement-only: does not change retrieval or truth resolution. Runs the full v2 path (multi-hop-aware), one clean API call per measurement — no exception/retry overhead.

Run it **after** the demos. `repeats=2` is enough for README numbers.

In [ ]:
def benchmark_hydramem_latency(
    database: str,
    queries: Optional[List[str]] = None,
    repeats: int = 2,
) -> pd.DataFrame:
    """Measure end-to-end live query latency on the v2 (multi-hop-aware) path."""
    queries = queries or [
        "Where does the user live?",
        "What does the user prefer?",
        "What is the user's favorite pet?",
        "Where does my manager live?",
    ]
    rows = []
    for query in queries:
        samples = []
        result = None
        for _ in range(max(1, repeats)):
            t0 = time.perf_counter()
            result = answer_memory_query_v2(query, database)
            samples.append((time.perf_counter() - t0) * 1000.0)
        samples.sort()
        rows.append({
            "query": query,
            "state": result["state"],
            "answer": result["answer"],
            "retrieved_candidates": result["retrieved_candidates"],
            "p50_ms": samples[len(samples) // 2],
            "min_ms": samples[0],
            "max_ms": samples[-1],
        })
    return pd.DataFrame(rows)




In [ ]:
# Example (run only when the live HydraDB database is ready):

latency_results = benchmark_hydramem_latency(DATABASE, repeats=2)
display(latency_results)
print(f"Overall p50: {latency_results['p50_ms'].median():.1f} ms")

,query,state,answer,retrieved_candidates,p50_ms,min_ms,max_ms
0,Where does the user live?,ANSWERABLE,London,13,5084.784099,2151.072334,5084.784099
1,What does the user prefer?,ANSWERABLE,dark mode,13,4938.521947,3742.630674,4938.521947
2,What is the user's favorite pet?,NO_EVIDENCE,None,13,2843.280594,2639.087649,2843.280594
3,Where does my manager live?,ANSWERABLE,Tokyo,13,3517.293790,3124.170790,3517.293790


Overall p50: 4227.9 ms


## 11d. Write-cost instrumentation

The Track 3 rubric asks for **"Read and write cost that would survive real usage."** Section 11c covers reads; this cell covers writes: ingest API time + graph-indexing time per fact.

Note: this ingests real facts (`predicate=benchmarked_in`) into the demo database. Run it **after** all demos; the facts are harmless noise for the resolver (different predicate), but "candidates retrieved" counts will include them.

In [ ]:
def benchmark_write_cost(database: str, n_facts: int = 5) -> Dict[str, float]:
    """Measure ingest + graph-indexing latency per fact."""
    test_facts = [
        make_fact(f"write_bench_{i}", 900 + i, i, "user", "benchmarked_in",
                  f"City_{i}", text=f"Benchmarking write cost in City_{i}.")
        for i in range(n_facts)
    ]

    # 1. Ingest API time
    t0 = time.perf_counter()
    ids = ingest_facts_with_graph(test_facts, database)
    write_ms = (time.perf_counter() - t0) * 1000.0

    # 2. Graph indexing time
    t0 = time.perf_counter()
    wait_for_indexing(ids, database, timeout_seconds=60)
    index_ms = (time.perf_counter() - t0) * 1000.0

    print(f"Write cost ({n_facts} facts):")
    print(f"  Ingest API call : {write_ms:.1f} ms ({write_ms / n_facts:.1f} ms/fact)")
    print(f"  Graph indexing  : {index_ms:.1f} ms ({index_ms / n_facts:.1f} ms/fact)")
    print(f"  Total to ready  : {write_ms + index_ms:.1f} ms")
    return {"ingest_total_ms": write_ms, "indexing_total_ms": index_ms}




In [ ]:
# ---------------------------------------------------------------------------
# Run BOTH benchmarks here, after the demos:
# ---------------------------------------------------------------------------
latency_results = benchmark_hydramem_latency(DATABASE, repeats=2)
display(latency_results)
print(f"Overall p50: {latency_results['p50_ms'].median():.1f} ms")
benchmark_write_cost(DATABASE, n_facts=5)

,query,state,answer,retrieved_candidates,p50_ms,min_ms,max_ms
0,Where does the user live?,ANSWERABLE,London,13,2931.614732,2901.354797,2931.614732
1,What does the user prefer?,ANSWERABLE,dark mode,13,3474.630014,2320.841781,3474.630014
2,What is the user's favorite pet?,NO_EVIDENCE,None,13,6216.524090,2496.578378,6216.524090
3,Where does my manager live?,ANSWERABLE,Tokyo,13,3023.770380,2642.461597,3023.770380


Overall p50: 3249.2 ms
Ingested 5 memories with BYOG graph.
Indexing completed (graph ready).
Write cost (5 facts):
  Ingest API call : 723.9 ms (144.8 ms/fact)
  Graph indexing  : 17140.6 ms (3428.1 ms/fact)
  Total to ready  : 17864.6 ms


{'ingest_total_ms': 723.940932000005, 'indexing_total_ms': 17140.642308999988}

In [ ]:
# ===========================================================================
# END-TO-END BENCHMARK: full corpus through HydraDB
# ingest -> index -> hybrid retrieval -> chunk parsing -> resolution
# ===========================================================================
E2E_DB = {
    "main":     "hydramem_bench_main",
    "conflict": "hydramem_bench_conflict",
    "open":     "hydramem_bench_open",
}

def run_end_to_end_benchmark():
    bench_conflict = [
        make_fact("bench_conflict_a", 100, 50, "user", "lives_in", "Boston",
                  text="I live in Boston now."),
        make_fact("bench_conflict_b", 101, 50, "user", "lives_in", "Cambridge",
                  text="I actually live in Cambridge now."),
    ]
    open_extras = [
        make_fact("favorite_color_blue", 200, 20, "user", "favorite_color", "blue",
                  fact_type="static", text="My favorite color is blue."),
        make_fact("pet_max", 201, 21, "user", "has_pet", "Max",
                  fact_type="static", text="My pet is named Max."),
        make_fact("salary_180k", 202, 22, "user", "salary", "$180k",
                  fact_type="static", text="My salary is $180k."),
    ]

    # --- 1. Ingest the full corpus (longer timeout: 42 facts x ~3.4s indexing) ---
    print("[1/2] Ingesting benchmark corpus into HydraDB...")
    ensure_database_ready(E2E_DB["main"])
    ids = ingest_facts_with_graph(benchmark_facts, E2E_DB["main"])
    wait_for_indexing(ids, E2E_DB["main"], timeout_seconds=600)

    ensure_database_ready(E2E_DB["conflict"])
    ids = ingest_facts_with_graph(bench_conflict, E2E_DB["conflict"])
    wait_for_indexing(ids, E2E_DB["conflict"], timeout_seconds=600)

    ensure_database_ready(E2E_DB["open"])
    ids = ingest_facts_with_graph(benchmark_facts + open_extras, E2E_DB["open"])
    wait_for_indexing(ids, E2E_DB["open"], timeout_seconds=600)

    # --- 2. Run all 11 cases through the LIVE query path ---
    print("[2/2] Running cases against live HydraDB...")
    cases = [
        ("current location", "Where does the user live?", "Boston", "temporal", E2E_DB["main"]),
        ("current job", "What is the user's job?", "researcher", "overwrite", E2E_DB["main"]),
        ("static preference", "What does the user prefer?", "dark mode", "static", E2E_DB["main"]),
        ("missing fact abstention", "What is the user's favorite pet?", None, "abstention", E2E_DB["main"]),
        ("property isolation (35 noise sessions)", "Where does the user live?", "Boston", "property-isolation", E2E_DB["main"]),
        ("explicit concurrent conflict", "Where does the user live?", None, "conflict", E2E_DB["conflict"]),
        ("revision chain intact", "Where does the user live?", "Boston", "revision-chain", E2E_DB["main"]),
        ("unknown property abstention", "What is the user's favorite pet?", None, "no-evidence", E2E_DB["main"]),
        ("open favorite color", "What is the user's favorite color?", "blue", "open-predicate", E2E_DB["open"]),
        ("open pet property", "What pet does the user have?", "Max", "open-predicate", E2E_DB["open"]),
        ("open salary property", "What is the user's salary?", "$180k", "open-predicate", E2E_DB["open"]),
    ]

    rows = []
    for name, query, expected, case_type, db in cases:
        t0 = time.perf_counter()
        # top_k=50: with a 42-fact corpus, default top_k=20 risks truncating
        # same-property candidates before the resolver can see the full chain.
        result = answer_memory_query(query, db, top_k=50)
        elapsed_ms = (time.perf_counter() - t0) * 1000.0

        if case_type == "conflict":
            passed = result["state"] == MemoryState.CONFLICTING.value
        elif expected is None:
            passed = result["state"] == MemoryState.NO_EVIDENCE.value
        else:
            passed = (result["answer"] == expected
                      and result["state"] == MemoryState.ANSWERABLE.value)

        rows.append({
            "case": name, "type": case_type, "target": result["target"],
            "expected": expected, "actual": result["answer"], "state": result["state"],
            "candidates_retrieved": result["retrieved_candidates"],
            "latency_ms": round(elapsed_ms, 1), "passed": passed,
        })

    return pd.DataFrame(rows)


e2e_results = run_end_to_end_benchmark()
display(e2e_results)
print(f"\nEnd-to-end accuracy: {e2e_results['passed'].mean():.1%} "
      f"({e2e_results['passed'].sum()}/{len(e2e_results)})")
print(f"Live latency p50: {e2e_results['latency_ms'].median():.0f} ms")
assert e2e_results["passed"].all(), "End-to-end benchmark has failing cases — inspect which retrieval/resolution step broke."

[1/2] Ingesting benchmark corpus into HydraDB...
Database ready: hydramem_bench_main
Ingested 42 memories with BYOG graph.
Indexing completed (graph ready).
Database ready: hydramem_bench_conflict
Ingested 2 memories with BYOG graph.
Indexing completed (graph ready).
Database ready: hydramem_bench_open
Ingested 45 memories with BYOG graph.
Indexing completed (graph ready).
[2/2] Running cases against live HydraDB...


,case,type,target,expected,actual,state,candidates_retrieved,latency_ms,passed
0,current location,temporal,user::lives_in,Boston,Boston,ANSWERABLE,42,3761.9,True
1,current job,overwrite,user::job,researcher,researcher,ANSWERABLE,42,3622.6,True
2,static preference,static,user::prefers,dark mode,dark mode,ANSWERABLE,42,5381.0,True
3,missing fact abstention,abstention,user::has_pet,None,None,NO_EVIDENCE,42,6084.4,True
4,property isolation (35 noise sessions),property-isolation,user::lives_in,Boston,Boston,ANSWERABLE,42,2546.6,True
5,explicit concurrent conflict,conflict,user::lives_in,None,None,CONFLICTING,2,5227.8,True
6,revision chain intact,revision-chain,user::lives_in,Boston,Boston,ANSWERABLE,42,2837.4,True
7,unknown property abstention,no-evidence,user::has_pet,None,None,NO_EVIDENCE,42,3682.4,True
8,open favorite color,open-predicate,user::favorite_color,blue,blue,ANSWERABLE,45,2924.9,True
9,open pet property,open-predicate,user::has_pet,Max,Max,ANSWERABLE,45,5114.8,True



End-to-end accuracy: 100.0% (11/11)
Live latency p50: 3762 ms


## 12. Status & remaining work (honest mapping)

> Outputs in this copy are intentionally blank. **Restart & Run All** to regenerate live outputs against real HydraDB.

### Track 3 brief — "What strong work shows"

| Requirement | Status | Evidence |
|-------------|--------|----------|
| Facts spread across many sessions / chronological reasoning | ✅ Solid | `resolve_revision_state` walks the SUPERSEDES chain by timestamp; synthetic benchmark covers 35 sessions |
| Information that changes over time / overwritten facts | ✅ Solid, live-verified | DEMO 1 live against real HydraDB: NYC→SF→London resolves correctly to London |
| Missing information / correct abstention | ✅ Solid | `NO_EVIDENCE` is a first-class enum state; benchmark tests missing-pet / unknown-property questions |
| A memory graph that makes time and revisions explicit | ✅ Solid, doc-confirmed | BYOG SUPERSEDES edges tagged `origin: "byog"` server-side — genuinely ours, not auto-extracted |
| Accuracy that a vector store alone cannot reach | ✅ Improved | Independent ground truth in benchmark; multi-hop (DEMO 4) is the strongest structural argument |
| Read and write cost that would survive real usage | ✅ Instrumented | Read: Section 11c (v2 path, clean single API calls). Write: Section 11d. Run both and paste numbers into the README |

### v2.2 hardening applied
- `hydra_query` probes the SDK signature once and filters kwargs up front — no per-call exceptions, no warning spam
- Unsupported temporal kwargs are no longer sent (SDK rejects them; resolver never reads the response fields)
- Word-boundary query targeting (no "deliver"/"believe" false positives)
- Clean extraction values: `'San Francisco'` and `'dark mode'`, no trailing noise
- Same-timestamp contradictions surface as `CONFLICTING` through the live extraction path (proof cell in Section 11)
- Chunk parse failures are logged and counted, never silent
- `reset_database()` helper for idempotent re-runs
- Benchmark oracles use hand-authored ground truth, not `max(timestamp)`

### Known limitations (disclose in README)
- Single-subject scoping in the single-hop path (`subject="user"`); multi-hop covers the manager/teammate cases
- TF-IDF baseline is a structural-failure proxy, not a tuned embedding model; the conflict-case baseline verdict is a documented design judgment
- Rule-based extractor covers high-frequency predicates; messy phrasing routes to the LLM path (Section 11b, wired but not run live in this notebook)

### Submission blockers (from Participant Guide)
- [ ] Public GitHub repo (open-source license, commits ≥ Aug 12)
- [ ] README explains project + how HydraDB is used + performance numbers
- [ ] 3-minute demo video (problem → project → live demo → why HydraDB matters)
- [ ] Official Google Form by **Aug 20, 11:59 PM PT**